# Silver Layer – Sales Details

This notebook processes sales data from the Bronze layer and prepares it for the Silver layer by applying data cleansing, validation, standardization, and transformation rules.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.table('workspace.bronze.crm_sales_details') 

## Clean Order Dates

Replace invalid order dates with null values and convert valid dates to date format.

In [0]:
df = df.withColumn('sls_order_dt' ,
               when(length(col("sls_order_dt").cast('string')) != 8 , None)
               .otherwise(to_date(col("sls_order_dt").cast('string') , 'yyyyMMdd'))
             )

## Standardize Date Columns

Convert shipping and due date fields from integer format to date format.

In [0]:
date_cols =  ['sls_ship_dt', 'sls_due_dt']
for c in date_cols :
    df = df.withColumn(c,
                       to_date(col(c).cast('string')  , 'yyyyMMdd')
    )

## Validate Sales Amount

Correct missing, invalid, or inconsistent sales amounts using quantity and price values.

In [0]:
df = df.withColumn( 'sls_sales' , 
              when(  
                   (col("sls_sales").isNull())  
                   | (col("sls_sales") <= 0)  
                   | ( col("sls_sales") !=  (col("sls_quantity") * abs(col("sls_price") ) )  )
                   , (col("sls_quantity") * abs(col("sls_price")) ) ) 
                   .otherwise(col("sls_sales")) 
)

## Validate Product Price

Calculate missing or invalid product prices using sales amount and quantity.

In [0]:
df = df.withColumn('sls_price' , 
              when(  (col("sls_price").isNull() )
                   | (col("sls_price")  <= 0 )
                   , (col("sls_sales") / col("sls_quantity") ) )
              .otherwise(col("sls_price"))
              )

## Rename Columns

Rename columns to follow the Silver layer naming convention.

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Preview Transformed Data

Review the transformed dataset before loading it into the Silver layer.

In [0]:
df.limit(10).display()

## Load to Silver Layer

Write the cleaned and transformed sales data to the Silver layer.

In [0]:
df.write.mode('overwrite').saveAsTable("workspace.silver.crm_sales")

## Verify Data Load

Validate that the sales data has been successfully loaded into the Silver layer.

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_sales LIMIT 10